# Tunisian apartment prices: every preprocessing step inside a Pipeline

The earlier notebooks scaled features, one-hot encoded cities and filled missing values on the **whole** dataset before splitting into train and test. That lets statistics from the test rows leak into training, so the reported scores are optimistic.

Here every fitted step (imputation, scaling, one-hot encoding, the log transform of the target and the model) lives inside one scikit-learn `Pipeline`. Cross-validation then refits it on each training fold only, and the saved model takes raw inputs (`superficie`, `chambres`, `bains`, `city`) directly.

Data: `data/tayara_cleanedV3.csv`, 1,370 apartment listings for sale scraped from Tayara.

In [1]:
import joblib
import numpy as np
import pandas as pd
from sklearn.compose import ColumnTransformer, TransformedTargetRegressor
from sklearn.ensemble import RandomForestRegressor
from sklearn.impute import SimpleImputer
from sklearn.linear_model import Ridge
from sklearn.model_selection import GridSearchCV, KFold, cross_val_score, train_test_split
from sklearn.metrics import mean_absolute_error, r2_score
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import OneHotEncoder, StandardScaler

df = pd.read_csv("../data/tayara_cleanedV3.csv", index_col=0)
df["city"] = df["city"].str.strip().str.lower()
X, y = df[["superficie", "chambres", "bains", "city"]], df["price"]
print(df.shape)
df.head()

(1370, 5)


,price,superficie,chambres,bains,city
1,430000.0,132.0,3,1,ariana
12,330000.0,90.0,2,1,ariana
16,210000.0,160.0,2,1,nabeul
18,315000.0,41.0,1,1,tunis
20,160000.0,124.0,3,1,sousse


## Hold out a test set before anything is fitted

Only row-wise cleaning (lower-casing city names) happened above; nothing has learned from the data yet.

In [2]:
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42)
cv = KFold(n_splits=5, shuffle=True, random_state=42)
len(X_train), len(X_test)

(1096, 274)

## The pipeline

- numeric columns: median imputation, then scaling (scaling only matters for the linear model);
- `city`: most-frequent imputation, then one-hot encoding that groups rare cities and ignores unseen ones at prediction time;
- target: prices span 65k to 4M DT, so the model learns `log(price)` and predictions are converted back automatically.

In [3]:
numeric = ["superficie", "chambres", "bains"]
categorical = ["city"]

preprocess = ColumnTransformer([
    ("num", Pipeline([("impute", SimpleImputer(strategy="median")),
                      ("scale", StandardScaler())]), numeric),
    ("cat", Pipeline([("impute", SimpleImputer(strategy="most_frequent")),
                      ("onehot", OneHotEncoder(handle_unknown="infrequent_if_exist", min_frequency=10))]), categorical),
])

def make_pipeline(model):
    return Pipeline([
        ("prep", preprocess),
        ("model", TransformedTargetRegressor(regressor=model, func=np.log1p, inverse_func=np.expm1)),
    ])

models = {
    "Ridge": make_pipeline(Ridge(alpha=1.0)),
    "RandomForest": make_pipeline(RandomForestRegressor(n_estimators=300, min_samples_leaf=2, random_state=42, n_jobs=-1)),
}
for name, pipe in models.items():
    r2 = cross_val_score(pipe, X_train, y_train, cv=cv, scoring="r2")
    mae = -cross_val_score(pipe, X_train, y_train, cv=cv, scoring="neg_mean_absolute_error")
    print(f"{name:<13} CV R2 {r2.mean():.3f} ± {r2.std():.3f}   CV MAE {mae.mean():,.0f} DT")

Ridge         CV R2 0.249 ± 0.048   CV MAE 167,652 DT


RandomForest  CV R2 0.241 ± 0.053   CV MAE 170,642 DT


## For comparison: the leaky way

The old approach: one-hot encode and scale **all rows**, then cross-validate. With this dataset the gap is small, because scaling and encoding learn little from the target; the point is that the pipeline score is the one you can trust, and the same pattern becomes a large error with target encoding, feature selection or oversampling (see Day 6 of the plan).

In [4]:
X_all = pd.get_dummies(X_train, columns=["city"], drop_first=True).astype(float)
X_all[numeric] = StandardScaler().fit_transform(X_all[numeric])  # fitted on every row, including the validation folds
leaky = TransformedTargetRegressor(regressor=RandomForestRegressor(n_estimators=300, min_samples_leaf=2, random_state=42, n_jobs=-1),
                                   func=np.log1p, inverse_func=np.expm1)
print("Leaky RF CV R2   :", cross_val_score(leaky, X_all, y_train, cv=cv, scoring="r2").mean().round(3))
print("Pipeline RF CV R2:", cross_val_score(models["RandomForest"], X_train, y_train, cv=cv, scoring="r2").mean().round(3))

Leaky RF CV R2   : 0.238


Pipeline RF CV R2: 0.241


## Tune the random forest inside the pipeline

`GridSearchCV` refits the whole pipeline on every fold, so tuning does not leak either.

In [5]:
grid = GridSearchCV(
    models["RandomForest"],
    {"model__regressor__min_samples_leaf": [1, 2, 4], "model__regressor__max_features": [1.0, 0.5]},
    cv=cv, scoring="r2", n_jobs=-1,
)
grid.fit(X_train, y_train)
print("best params:", grid.best_params_, " CV R2:", round(grid.best_score_, 3))

best params: {'model__regressor__max_features': 0.5, 'model__regressor__min_samples_leaf': 4}  CV R2: 0.26


## Final check on the untouched test set

In [6]:
best = grid.best_estimator_
pred = best.predict(X_test)
print(f"Test R2  {r2_score(y_test, pred):.3f}")
print(f"Test MAE {mean_absolute_error(y_test, pred):,.0f} DT")
print(f"Median absolute % error {np.median(np.abs(pred - y_test) / y_test) * 100:.1f}%")

Test R2  0.172
Test MAE 216,341 DT
Median absolute % error 26.2%


## Save one artifact

The pipeline replaces the separate `scaler.pkl`, `model_columns.pkl` and model files: it takes a raw DataFrame and returns a price.

In [7]:
joblib.dump(best, "../models/price_pipeline.joblib")
sample = pd.DataFrame([{"superficie": 120, "chambres": 3, "bains": 1, "city": "ariana"},
                       {"superficie": 90, "chambres": 2, "bains": 1, "city": "a city never seen"}])
joblib.load("../models/price_pipeline.joblib").predict(sample).round(-3)

array([239000., 199000.])

## What the honest numbers say

With every step inside the pipeline, the best model explains about a quarter of the price variance (CV R² ≈ 0.26, test R² ≈ 0.17) and is off by a median of about 26%. Four columns (surface, rooms, bathrooms, city) are not enough to price an apartment: neighbourhood within the city, floor, age, standing and parking matter a lot. Next steps: extract those from the listing text, and group the cross-validation by listing duplicates so reposted ads cannot sit on both sides of a split.